## Inter-Species Dose Conversion (BSA Method)

In [39]:
# TODO: Ready to go - send to Immanuel and Omar.

# This (below) is a look-up table for body weight (wt, kilograms), body surface area (SA, meters-squared), and km factor ((body weight) / (body surface area)) in that order. Values for km differ slightly from the actual calculation using body weight and body surface area b/c they were obtained from FDA guidance, and not from calculation.

#               Species              Wt         SA          km
#-------------------------------------------------------------------
wt_sa_km_lut = {'mouse':            [0.02,      0.007,      3],
                'rat':              [0.15,      0.025,      6],
                'dog':              [10,        0.50,       20],
                'monkey':           [3,         0.25,       12],
                'human':            [60,        1.62,       37],
                'rabbit':           [1.8,       0.15,       12],
                'child':            [20,        0.8,        25],
                'hamster':          [0.08,      0.016,      5],
                'ferret':           [0.3,       0.043,      7],
                'guinea pig':       [0.4,       0.05,       8],
                'marmoset':         [0.35,      0.06,       6],
                'squirrel monkey':  [0.6,       0.09,       7],
                'baboon':           [12,        0.6,        20],
                'micropig':         [20,        0.74,       27],
                'minipig':          [40,        1.14,       35]}

In [40]:
def calc_eq_dose(param_dict):
    """
    Main function that does all the stuff.
    :param param_dict: Dictionary of parameter values.
    :return: Nothing - just calls a function that prints the result.
    """
    param_dict = define_wt_km(param_dict)

    param_dict['dose_b'] = round((param_dict['dose_a'] * param_dict['sp_a_km']) / param_dict['sp_b_km'])
    param_dict['bsa_dose'] = round(get_per_m2_dose(param_dict))

    print_result(param_dict)


def define_wt_km(param_dict):
    """
    Recalculates km factors only if new wt or sa values are defined. Otherwise keeps the FDA-defined values for km, which slightly differ from what would be calculated from wt and sa. Global variable 'fda_defined' becomes false if user has defined a custom body weight.
    :param param_dict: parameter dictionary
    :return: parameter dictionary with lookup or calculated km factors.
    """
    global fda_compliant

    sp_a, sp_b = param_dict['species_a'], param_dict['species_b']
    sp_a_wt, sp_a_sa, sp_a_km = wt_sa_km_lut[sp_a]
    sp_b_wt, sp_b_sa, sp_b_km = wt_sa_km_lut[sp_b]

    # Recalculate km only if user defined custom body weight(s).
    if param_dict['sp_a_wt']:
        param_dict['sp_a_km'] = param_dict['new_sp_a_wt'] / sp_a_sa
        fda_compliant = False
    else:
        param_dict['sp_a_wt'] = sp_a_wt
        param_dict['sp_a_km'] = sp_a_km
    param_dict['sp_a_sa'] = sp_a_sa

    if param_dict['sp_b_wt']:
        param_dict['sp_b_km'] = param_dict['sp_b_wt'] / sp_b_sa
        fda_compliant = False
    else:
        param_dict['sp_b_wt'] = sp_b_wt
        param_dict['sp_b_km'] = sp_b_km
    param_dict['sp_b_sa'] = sp_b_sa

    return param_dict


def get_per_m2_dose(param_dict):
    """
    Simple function that just returns a body-surface-area (BSA) dose calculated for 'species_a'.
    :param param_dict: Dictionary of parameter values.
    :return: A body-surface-area dose calculated for 'species_a'.
    """
    return param_dict['dose_a'] * param_dict['sp_a_km']


def print_result(param_dict):
    """
    Formats calculation result for printing.
    :param param_dict: Dictionary of parameter values.
    :return: Nothing - just prints the result.
    """
    sp_a, sp_b =    param_dict['species_a'],    param_dict['species_b']
    km_a, km_b =    param_dict['sp_a_km'],      param_dict['sp_b_km']
    d_a, d_b =      param_dict['dose_a'],       param_dict['dose_b']
    bsa_dose =      param_dict['bsa_dose']

    print(f'{sp_a + " dose (mg/kg):":<30} {d_a:<10}')
    print(f'{sp_a + " km factor (kg/m2):":<30} {km_a:<10}\n')
    print(f'{"BSA dose (mg/m2):":<30} {round(bsa_dose):<10}\n')

    print(f'{sp_b + " km factor (kg/m2):":<30} {km_b:<10}')
    print(f'{sp_b + " dose (mg/kg):":<30} {round(d_b):<10}\n')

    if sp_b == 'human':
        compliance_phrase = {True: "complies", False: "does NOT comply"}[fda_compliant]
        print(f"This corresponds to an absolute human equivalent dose (HED) of {round(d_b * wt_sa_km_lut[sp_b][0])} mg. No safety factor has been applied. This output {compliance_phrase} with values defined in FDA guidance for calculating safe starting dose in healthy volunteers.")

In [41]:
fda_compliant = True    # Becomes false if user defines a custom animal weight.

calc_params = {'species_a':         'dog',      # Input label "Species A", choose from drop-down list comprising all keys in wt_sa_km_lut dictionary.
               'dose_a':            200,        # Input label "Species A Dose". Text box, enforce numeric.
               'species_b':         'human',    # Input label "Species B", choose from drop-down list comprising all keys in wt_sa_km_lut dictionary.

             # Lookup values that can be overridden by user input:
               'sp_a_wt':           None,       # Input label "Species A Weight (kg, optional)". Text box, enforce numeric.
               'sp_b_wt':           None,       # Input label "Species B Weight (kg, optional)". Text box, enforce numeric.

             # Lookup and calculated values, cannot be overridden by user input:
               'sp_a_sa':           None,
               'sp_b_sa':           None,
               'sp_a_km':           None,
               'sp_b_km':           None,
               'dose_b':            None,
               'bsa_dose':          None,
               'human_dose':        None}

# Button labeled "Calculate" calls the function below.
calc_eq_dose(calc_params)

dog dose (mg/kg):              200       
dog km factor (kg/m2):         20        

BSA dose (mg/m2):              4000      

human km factor (kg/m2):       37        
human dose (mg/kg):            108       

This corresponds to an absolute human equivalent dose (HED) of 6480 mg. No safety factor has been applied. This output complies with values defined in FDA guidance for calculating safe starting dose in healthy volunteers.
